# Citation Verifier

## Check Whether Generated Answers Are Truly Supported by Retrieved Chunks

**Project 24** - Advance RAG Learning Series

| Property | Value |
|----------|-------|
| Task | Groundedness verification of RAG-generated answers |
| Method | Embedding-based claim-to-evidence matching |
| Retrieval | Dense embedding search (sentence-transformers) |
| Corpus | 25-document Solar System knowledge base |
| Verdicts | SUPPORTED, PARTIAL, UNSUPPORTED |
| Evaluation | Groundedness score, hallucination rate, per-claim breakdown |


## Project Overview

RAG reduces hallucination compared to vanilla LLMs, but doesn't eliminate it.
The generator can still:

- **Blend** retrieved facts with parametric (memorized) knowledge
- **Misinterpret** numbers, dates, or relationships from context
- **Generate plausible-sounding claims** not in any retrieved chunk

A **Citation Verifier** catches these failures *before* the answer reaches
the user. For each factual claim in the answer, it checks whether the
retrieved chunks actually support that claim.

### Pipeline

```
Question --> Retrieve chunks --> Generate answer --> Extract claims
                                                        |
                                              For each claim:
                                                Match against chunks
                                                Assign verdict
                                                        |
                                              Groundedness report
```


## Learning Goals

1. Understand **why** RAG answers still need verification
2. Implement claim extraction from generated answers
3. Build an embedding-based groundedness checker
4. Classify claims as SUPPORTED, PARTIAL, or UNSUPPORTED
5. Compute a **groundedness score** for the entire answer
6. Analyze common hallucination patterns in RAG


## Problem Statement

Given:
- A question
- A set of retrieved context chunks
- A generated answer

Verify whether each factual claim in the answer is actually supported
by the retrieved chunks. Produce a **groundedness report** with:
- Per-claim verdicts (SUPPORTED / PARTIAL / UNSUPPORTED)
- Evidence citations for each claim
- Overall groundedness score


## Why Citation Verification Matters

| Problem | Impact | Solution |
|---------|--------|----------|
| Users trust RAG answers without checking | Misinformation propagates | Auto-verify before showing answer |
| LLMs mix retrieved + memorized knowledge | Subtle hallucinations | Flag claims not grounded in chunks |
| Compliance/legal domains need traceability | Regulatory risk | Every claim must cite its source |
| Long context = more room for error | Hard to manually audit | Automated claim-by-claim checking |

**Citation verification is the last line of defense in a RAG pipeline.**


## Environment Setup

In [ ]:
import subprocess, sys, warnings

def _install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for pkg in ["sentence-transformers"]:
    try:
        __import__(pkg.replace("-", "_"))
    except ImportError:
        _install(pkg)

warnings.filterwarnings("ignore")
print("Environment ready.")


## Imports

In [ ]:
import re, random, textwrap
from typing import List, Dict, Tuple
from dataclasses import dataclass, field

import numpy as np
from sentence_transformers import SentenceTransformer

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print("All imports loaded.")


## Configuration

In [ ]:
EMBEDDING_MODEL = "all-MiniLM-L6-v2"
K = 3                # top-K chunks to retrieve per question
SUPPORT_THRESHOLD = 0.70    # cosine sim >= this -> SUPPORTED
PARTIAL_THRESHOLD = 0.50    # cosine sim >= this -> PARTIAL
# Below PARTIAL_THRESHOLD -> UNSUPPORTED

print(f"Config: model={EMBEDDING_MODEL}, K={K}")
print(f"  SUPPORTED threshold >= {SUPPORT_THRESHOLD}")
print(f"  PARTIAL threshold   >= {PARTIAL_THRESHOLD}")
print(f"  UNSUPPORTED         <  {PARTIAL_THRESHOLD}")


## Knowledge Base

A 25-chunk factual corpus about the Solar System.
Each chunk is a self-contained paragraph about one topic.
This lets us precisely verify which facts come from which chunks.


In [ ]:
corpus = [
    {"id": "C01", "text": "Mercury is the smallest planet in the Solar System and closest to the Sun. Its surface temperature ranges from -180 degrees Celsius at night to 430 degrees Celsius during the day. Mercury has no atmosphere and no moons. Its orbital period is 88 Earth days."},
    {"id": "C02", "text": "Venus is the hottest planet due to its thick carbon dioxide atmosphere creating a runaway greenhouse effect. Surface temperature averages 465 degrees Celsius. Venus rotates backward in retrograde rotation. Venus has no moons."},
    {"id": "C03", "text": "Earth is the only known planet with liquid water on its surface. Average temperature is 15 degrees Celsius. Earth has one natural satellite called the Moon. The Moon is 384,400 km away and affects Earth tidal patterns."},
    {"id": "C04", "text": "Mars is called the Red Planet due to iron oxide on its surface. Mars has two small moons named Phobos and Deimos. Surface temperature averages minus 65 degrees Celsius. Mars has the tallest volcano in the Solar System, Olympus Mons, at 21.9 km high."},
    {"id": "C05", "text": "Jupiter is the largest planet with a mass of 1.898 times 10 to the 27 kg. Jupiter has at least 95 known moons, including the four Galilean moons Io, Europa, Ganymede, and Callisto. Jupiter Great Red Spot is a storm larger than Earth that has persisted for at least 400 years."},
    {"id": "C06", "text": "Saturn is famous for its extensive ring system made primarily of ice particles and rocky debris. Saturn has 146 known moons, with Titan being the largest. Titan has a thick nitrogen atmosphere and liquid methane lakes on its surface."},
    {"id": "C07", "text": "Uranus rotates on its side with an axial tilt of 97.77 degrees. It has 28 known moons and a faint ring system. Uranus was the first planet discovered with a telescope by William Herschel in 1781. Its atmosphere contains methane which gives it a blue-green color."},
    {"id": "C08", "text": "Neptune is the farthest planet from the Sun. It has the strongest winds in the Solar System reaching up to 2,100 km per hour. Neptune has 16 known moons, with Triton being the largest. Triton orbits Neptune in a retrograde direction."},
    {"id": "C09", "text": "The Sun contains 99.86 percent of the total mass of the Solar System. Its core temperature is approximately 15 million degrees Celsius. The Sun is about 4.6 billion years old and is classified as a G-type main-sequence star. It will eventually become a red giant in about 5 billion years."},
    {"id": "C10", "text": "The asteroid belt lies between Mars and Jupiter orbits. It contains millions of rocky bodies. The largest object in the asteroid belt is Ceres, which is classified as a dwarf planet. The total mass of the asteroid belt is less than 4 percent of the Moon mass."},
    {"id": "C11", "text": "Pluto was reclassified from a planet to a dwarf planet in 2006 by the International Astronomical Union. Pluto has five known moons with Charon being the largest. The Pluto-Charon system is sometimes considered a double dwarf planet because Charon is relatively large compared to Pluto."},
    {"id": "C12", "text": "The Kuiper Belt is a region of icy bodies beyond Neptune orbit extending from about 30 to 50 AU from the Sun. It contains dwarf planets including Pluto, Haumea, and Makemake. The Kuiper Belt is the source of many short-period comets."},
    {"id": "C13", "text": "The Oort Cloud is a theoretical spherical shell of icy objects surrounding the Solar System at distances between 2,000 and 200,000 AU. It is the source of long-period comets. The Oort Cloud has never been directly observed but its existence is inferred from comet trajectories."},
    {"id": "C14", "text": "Europa, one of Jupiter Galilean moons, has a subsurface ocean beneath its icy crust. Scientists believe Europa ocean may contain more water than all of Earth oceans combined. Europa is considered one of the most promising places to search for extraterrestrial life in the Solar System."},
    {"id": "C15", "text": "The Solar System formed approximately 4.6 billion years ago from the gravitational collapse of a giant molecular cloud. Most of the collapsing mass collected in the center forming the Sun. The remaining material formed a protoplanetary disk from which the planets formed."},
    {"id": "C16", "text": "Mars has polar ice caps made of water ice and dry ice (frozen carbon dioxide). During Martian summer, the dry ice sublimates revealing the water ice underneath. The northern polar cap is about 1,000 km in diameter."},
    {"id": "C17", "text": "Jupiter magnetic field is the strongest of any planet, about 20,000 times stronger than Earth magnetic field. This creates intense radiation belts around Jupiter. The magnetosphere extends up to 7 million km toward the Sun."},
    {"id": "C18", "text": "Saturn density is the lowest of any planet at 0.687 grams per cubic centimeter, which is less than water. This means Saturn would theoretically float if placed in a sufficiently large body of water. Saturn orbital period is about 29.5 Earth years."},
    {"id": "C19", "text": "Titan, Saturn largest moon, is the only moon in the Solar System with a substantial atmosphere. Its atmospheric pressure at the surface is about 1.5 times that of Earth. Titan surface temperature is about minus 179 degrees Celsius. Cassini-Huygens mission landed a probe on Titan in January 2005."},
    {"id": "C20", "text": "The Voyager 1 spacecraft launched in 1977 is the most distant human-made object from Earth. It entered interstellar space in August 2012. Voyager 1 carries a golden record containing sounds and images of Earth for any extraterrestrial life that might find it."},
    {"id": "C21", "text": "Comets are icy bodies that develop a visible atmosphere called a coma when they approach the Sun. The coma and tail form as solar radiation and solar wind cause material to sublimate from the comet nucleus. Short-period comets originate from the Kuiper Belt while long-period comets come from the Oort Cloud."},
    {"id": "C22", "text": "The habitable zone, also called the Goldilocks zone, is the region around a star where conditions might be right for liquid water to exist on a planet surface. In our Solar System, the habitable zone extends roughly from 0.95 to 1.67 AU from the Sun. Earth is the only planet currently within this zone."},
    {"id": "C23", "text": "Io, the innermost Galilean moon of Jupiter, is the most volcanically active body in the Solar System. It has over 400 active volcanoes. The volcanism is caused by tidal heating from Jupiter gravitational pull. Io surface is constantly reshaped by lava flows."},
    {"id": "C24", "text": "The Great Dark Spot was a storm on Neptune observed by Voyager 2 in 1989. Unlike Jupiter Great Red Spot, Neptune storm disappeared within a few years. Neptune atmospheric composition is primarily hydrogen and helium with traces of methane."},
    {"id": "C25", "text": "Ganymede, the largest moon in the Solar System, is even larger than the planet Mercury. Ganymede orbits Jupiter and is one of the four Galilean moons. It is the only moon known to have its own magnetic field. Ganymede has a subsurface ocean similar to Europa."},
]

print(f"Knowledge base: {len(corpus)} chunks")
for doc in corpus[:3]:
    print(f"  [{doc['id']}] {doc['text'][:70]}...")
print("  ...")


## Dense Retriever

Encode all chunks with sentence-transformers. Retrieve top-K for each question.


In [ ]:
print(f"Loading embedding model: {EMBEDDING_MODEL}...")
encoder = SentenceTransformer(EMBEDDING_MODEL)

chunk_texts = [doc["text"] for doc in corpus]
chunk_embeddings = encoder.encode(chunk_texts, convert_to_numpy=True, show_progress_bar=False)
chunk_embeddings = chunk_embeddings / np.linalg.norm(chunk_embeddings, axis=1, keepdims=True)


def retrieve(query: str, k: int = K) -> List[Dict]:
    """Retrieve top-k chunks by cosine similarity."""
    q_emb = encoder.encode([query], convert_to_numpy=True)
    q_emb = q_emb / np.linalg.norm(q_emb, axis=1, keepdims=True)
    scores = (chunk_embeddings @ q_emb.T).flatten()
    top_idx = np.argsort(scores)[::-1][:k]
    return [{"id": corpus[i]["id"], "text": corpus[i]["text"],
             "score": float(scores[i])}
            for i in top_idx]


# Sanity check
test = retrieve("What are Jupiter moons?", k=3)
print(f"\nDense retriever ready (index shape: {chunk_embeddings.shape})")
print(f"Test query: 'What are Jupiter moons?'")
for r in test:
    print(f"  [{r['id']}] score={r['score']:.3f} | {r['text'][:60]}...")


## Claim Extraction

Break a generated answer into individual factual claims.
Each claim is a single verifiable statement.

In production, an LLM extracts claims. Here we use sentence splitting
with filtering to keep the notebook self-contained.

**Key insight:** Each sentence in a well-formed answer typically
contains one or two checkable facts.


In [ ]:
def extract_claims(answer: str) -> List[str]:
    """Extract individual factual claims from an answer.
    
    Splits on sentence boundaries, filters out non-factual content
    (questions, hedges, transitions).
    """
    # Split into sentences
    sentences = re.split(r'(?<=[.!?])\s+', answer.strip())
    
    claims = []
    for sent in sentences:
        sent = sent.strip()
        if len(sent) < 15:  # too short to be a factual claim
            continue
        # Skip non-factual sentences
        skip_patterns = [
            r"^(In summary|Overall|To summarize|In conclusion)",
            r"^(However|But|That said|Note that)",
            r"\?$",  # questions
            r"^(Yes|No|Sure|Of course)",
        ]
        if any(re.search(p, sent, re.IGNORECASE) for p in skip_patterns):
            continue
        claims.append(sent)
    
    return claims


# Demo
demo_answer = (
    "Mars is called the Red Planet due to iron oxide on its surface. "
    "It has two moons named Phobos and Deimos. "
    "Mars has the tallest volcano, Olympus Mons, at 21.9 km high. "
    "In summary, Mars is a fascinating planet."
)
claims = extract_claims(demo_answer)
print(f"Answer: {demo_answer[:80]}...")
print(f"\nExtracted {len(claims)} claims:")
for i, c in enumerate(claims, 1):
    print(f"  {i}. {c}")


## Groundedness Checker

For each claim, compute cosine similarity against every retrieved chunk.
The highest similarity determines the verdict:

| Similarity | Verdict | Meaning |
|-----------|---------|----------|
| >= 0.70 | SUPPORTED | Claim is directly stated in a chunk |
| >= 0.50 | PARTIAL | Related info exists but doesn't fully confirm |
| < 0.50 | UNSUPPORTED | No evidence in retrieved chunks |

**Why embedding similarity works for verification:**
- Claims that rephrase chunk content have high similarity
- Hallucinated claims about topics not in the chunks have low similarity
- Partial matches capture related-but-not-exact support


In [ ]:
@dataclass
class ClaimVerdict:
    """Verification result for a single claim."""
    claim: str
    verdict: str           # SUPPORTED, PARTIAL, UNSUPPORTED
    best_score: float      # highest similarity to any chunk
    best_chunk_id: str     # which chunk matched best
    best_chunk_text: str   # the chunk text (for citation)


@dataclass
class GroundednessReport:
    """Full verification report for an answer."""
    question: str
    answer: str
    chunks_used: List[Dict]
    claim_verdicts: List[ClaimVerdict]
    groundedness_score: float  # fraction of claims that are SUPPORTED
    full_support_score: float  # fraction SUPPORTED + PARTIAL


def verify_groundedness(question: str, answer: str,
                        chunks: List[Dict]) -> GroundednessReport:
    """Verify whether an answer is grounded in retrieved chunks."""
    claims = extract_claims(answer)
    if not claims:
        return GroundednessReport(
            question=question, answer=answer, chunks_used=chunks,
            claim_verdicts=[], groundedness_score=0.0, full_support_score=0.0
        )
    
    # Encode claims and chunks
    claim_embs = encoder.encode(claims, convert_to_numpy=True)
    claim_embs = claim_embs / np.linalg.norm(claim_embs, axis=1, keepdims=True)
    
    chunk_texts_local = [c["text"] for c in chunks]
    chunk_embs = encoder.encode(chunk_texts_local, convert_to_numpy=True)
    chunk_embs = chunk_embs / np.linalg.norm(chunk_embs, axis=1, keepdims=True)
    
    # Similarity matrix: claims x chunks
    sim_matrix = claim_embs @ chunk_embs.T
    
    verdicts = []
    for i, claim in enumerate(claims):
        best_j = int(np.argmax(sim_matrix[i]))
        best_score = float(sim_matrix[i, best_j])
        
        if best_score >= SUPPORT_THRESHOLD:
            verdict = "SUPPORTED"
        elif best_score >= PARTIAL_THRESHOLD:
            verdict = "PARTIAL"
        else:
            verdict = "UNSUPPORTED"
        
        verdicts.append(ClaimVerdict(
            claim=claim,
            verdict=verdict,
            best_score=best_score,
            best_chunk_id=chunks[best_j]["id"],
            best_chunk_text=chunks[best_j]["text"],
        ))
    
    supported = sum(1 for v in verdicts if v.verdict == "SUPPORTED")
    partial = sum(1 for v in verdicts if v.verdict == "PARTIAL")
    total = len(verdicts)
    
    return GroundednessReport(
        question=question,
        answer=answer,
        chunks_used=chunks,
        claim_verdicts=verdicts,
        groundedness_score=supported / total,
        full_support_score=(supported + partial) / total,
    )


print("Groundedness checker defined.")


## Report Printer

In [ ]:
ICONS = {"SUPPORTED": "+", "PARTIAL": "~", "UNSUPPORTED": "X"}

def print_report(report: GroundednessReport) -> None:
    """Print a formatted groundedness report."""
    print(f"Question: {report.question}")
    print(f"Answer: {report.answer[:120]}...")
    print(f"\nChunks retrieved: {[c['id'] for c in report.chunks_used]}")
    print(f"\n--- Claim-by-Claim Verification ---")
    
    for i, v in enumerate(report.claim_verdicts, 1):
        icon = ICONS[v.verdict]
        print(f"\n  [{icon}] Claim {i}: {v.claim}")
        print(f"      Verdict: {v.verdict} (score={v.best_score:.3f})")
        print(f"      Evidence: [{v.best_chunk_id}] {v.best_chunk_text[:80]}...")
    
    print(f"\n--- Summary ---")
    total = len(report.claim_verdicts)
    supported = sum(1 for v in report.claim_verdicts if v.verdict == "SUPPORTED")
    partial = sum(1 for v in report.claim_verdicts if v.verdict == "PARTIAL")
    unsupported = sum(1 for v in report.claim_verdicts if v.verdict == "UNSUPPORTED")
    print(f"  Total claims: {total}")
    print(f"  SUPPORTED:    {supported} ({supported/total:.0%})")
    print(f"  PARTIAL:      {partial} ({partial/total:.0%})")
    print(f"  UNSUPPORTED:  {unsupported} ({unsupported/total:.0%})")
    print(f"  Groundedness: {report.groundedness_score:.1%}")
    print(f"  Full support: {report.full_support_score:.1%}")
    print("=" * 70)


print("Report printer defined.")


## Test Scenarios

We define pre-written answers that simulate LLM outputs.
Some are fully grounded, some contain hallucinations.
This lets us precisely evaluate the verifier.

- **Test 1:** Fully grounded answer about Mars
- **Test 2:** Answer that mixes real facts with a hallucination
- **Test 3:** Multi-topic answer with facts from different chunks
- **Test 4:** Answer with plausible but fabricated statistics
- **Test 5:** Well-grounded answer about the outer Solar System


In [ ]:
test_scenarios = [
    {
        "question": "Tell me about Mars, its moons, and notable features.",
        "answer": (
            "Mars is called the Red Planet due to iron oxide on its surface. "
            "Mars has two small moons named Phobos and Deimos. "
            "Surface temperature averages minus 65 degrees Celsius. "
            "Mars has the tallest volcano in the Solar System, Olympus Mons, at 21.9 km high."
        ),
        "expected_verdicts": ["SUPPORTED", "SUPPORTED", "SUPPORTED", "SUPPORTED"],
        "label": "Fully grounded (all facts from C04)",
    },
    {
        "question": "What are Jupiter most notable features?",
        "answer": (
            "Jupiter is the largest planet with a mass of 1.898 times 10 to the 27 kg. "
            "Jupiter has at least 95 known moons including the Galilean moons. "
            "Jupiter Great Red Spot is a storm larger than Earth. "
            "Jupiter average surface temperature is minus 145 degrees Celsius and it rains diamonds in its atmosphere."
        ),
        "expected_verdicts": ["SUPPORTED", "SUPPORTED", "SUPPORTED", "UNSUPPORTED"],
        "label": "3 grounded + 1 hallucination (diamond rain not in corpus)",
    },
    {
        "question": "Compare the temperatures of planets in the inner Solar System.",
        "answer": (
            "Mercury surface temperature ranges from minus 180 degrees at night to 430 degrees during the day. "
            "Venus is the hottest planet with surface temperature averaging 465 degrees Celsius. "
            "Earth average temperature is 15 degrees Celsius. "
            "Mars surface temperature averages minus 65 degrees Celsius."
        ),
        "expected_verdicts": ["SUPPORTED", "SUPPORTED", "SUPPORTED", "SUPPORTED"],
        "label": "Cross-chunk grounded (facts from C01, C02, C03, C04)",
    },
    {
        "question": "Tell me about Europa and its potential for life.",
        "answer": (
            "Europa has a subsurface ocean beneath its icy crust. "
            "Scientists believe Europa ocean may contain more water than all of Earth oceans combined. "
            "Europa is considered one of the most promising places to search for extraterrestrial life. "
            "Recent satellite observations have detected organic molecules on Europa surface. "
            "NASA Clipper mission will study Europa habitability when it arrives in 2030."
        ),
        "expected_verdicts": ["SUPPORTED", "SUPPORTED", "SUPPORTED", "UNSUPPORTED", "UNSUPPORTED"],
        "label": "3 grounded + 2 hallucinations (organic molecules and Clipper details fabricated)",
    },
    {
        "question": "What do we know about the outer Solar System beyond Neptune?",
        "answer": (
            "The Kuiper Belt extends from about 30 to 50 AU from the Sun and contains icy bodies. "
            "It contains dwarf planets including Pluto, Haumea, and Makemake. "
            "The Oort Cloud is a theoretical spherical shell at distances between 2,000 and 200,000 AU. "
            "The Oort Cloud is the source of long-period comets."
        ),
        "expected_verdicts": ["SUPPORTED", "SUPPORTED", "SUPPORTED", "SUPPORTED"],
        "label": "Fully grounded (facts from C12 and C13)",
    },
]

print(f"Test scenarios: {len(test_scenarios)}")
for i, t in enumerate(test_scenarios, 1):
    print(f"  Test {i}: {t['label']}")


## Running the Citation Verifier

For each test scenario:
1. Retrieve relevant chunks for the question
2. Verify the pre-written answer against retrieved chunks
3. Print the groundedness report


In [ ]:
all_reports = []

for i, scenario in enumerate(test_scenarios, 1):
    print(f"\n{'=' * 70}")
    print(f"TEST {i}: {scenario['label']}")
    print(f"{'=' * 70}")
    
    # Retrieve chunks
    chunks = retrieve(scenario["question"], k=K)
    
    # Verify
    report = verify_groundedness(
        question=scenario["question"],
        answer=scenario["answer"],
        chunks=chunks,
    )
    all_reports.append(report)
    
    # Print
    print_report(report)


## Aggregate Verification Metrics

Compute overall performance across all test scenarios.


In [ ]:
print("Aggregate Metrics Across All Test Scenarios")
print("=" * 60)

total_claims = 0
total_supported = 0
total_partial = 0
total_unsupported = 0

for i, report in enumerate(all_reports, 1):
    n = len(report.claim_verdicts)
    s = sum(1 for v in report.claim_verdicts if v.verdict == "SUPPORTED")
    p = sum(1 for v in report.claim_verdicts if v.verdict == "PARTIAL")
    u = sum(1 for v in report.claim_verdicts if v.verdict == "UNSUPPORTED")
    total_claims += n
    total_supported += s
    total_partial += p
    total_unsupported += u
    print(f"  Test {i}: {s}S {p}P {u}U / {n} claims  "
          f"(groundedness={report.groundedness_score:.0%}) | {test_scenarios[i-1]['label'][:50]}")

print(f"\n  TOTAL: {total_supported}S {total_partial}P {total_unsupported}U / {total_claims} claims")
print(f"  Overall groundedness:   {total_supported/total_claims:.1%}")
print(f"  Full support (S+P):     {(total_supported+total_partial)/total_claims:.1%}")
print(f"  Hallucination rate:     {total_unsupported/total_claims:.1%}")


## Verifier Accuracy

Compare the verifier's verdicts against our ground-truth labels.
This tells us how well embedding similarity distinguishes
supported claims from hallucinations.


In [ ]:
print("Verifier Accuracy vs Expected Verdicts")
print("=" * 60)

correct = 0
total = 0
confusion = {"TP": 0, "FP": 0, "TN": 0, "FN": 0}

for i, (report, scenario) in enumerate(zip(all_reports, test_scenarios), 1):
    expected = scenario["expected_verdicts"]
    actual = [v.verdict for v in report.claim_verdicts]
    
    # Match length (take min if claim extraction differs)
    n = min(len(expected), len(actual))
    
    scenario_correct = 0
    for j in range(n):
        exp_supported = expected[j] == "SUPPORTED"
        act_supported = actual[j] in ("SUPPORTED", "PARTIAL")  # count PARTIAL as support
        
        if exp_supported and act_supported:
            confusion["TP"] += 1
            scenario_correct += 1
        elif not exp_supported and not act_supported:
            confusion["TN"] += 1
            scenario_correct += 1
        elif exp_supported and not act_supported:
            confusion["FN"] += 1  # missed real support
        else:
            confusion["FP"] += 1  # false support for hallucination
        total += 1
    
    correct += scenario_correct
    
    print(f"  Test {i}: expected={expected}")
    print(f"           actual  ={actual}")
    match = "MATCH" if expected[:n] == actual[:n] else f"{scenario_correct}/{n} correct"
    print(f"           -> {match}")
    print()

print(f"Overall accuracy: {correct}/{total} ({correct/total:.1%})")
print(f"\nConfusion (support detection):")
print(f"  TP (correctly identified supported):     {confusion['TP']}")
print(f"  TN (correctly identified hallucination): {confusion['TN']}")
print(f"  FP (missed hallucination):               {confusion['FP']}")
print(f"  FN (missed real support):                {confusion['FN']}")

precision = confusion["TP"] / (confusion["TP"] + confusion["FP"]) if (confusion["TP"] + confusion["FP"]) > 0 else 0
recall = confusion["TP"] / (confusion["TP"] + confusion["FN"]) if (confusion["TP"] + confusion["FN"]) > 0 else 0
f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
print(f"\n  Precision: {precision:.3f}")
print(f"  Recall:    {recall:.3f}")
print(f"  F1:        {f1:.3f}")


## Threshold Sensitivity Analysis

How do different SUPPORT/PARTIAL thresholds affect verdict distribution?
This helps calibrate the verifier for your use case.


In [ ]:
# Collect all (claim, best_score, expected_grounded) pairs
all_pairs = []
for report, scenario in zip(all_reports, test_scenarios):
    expected = scenario["expected_verdicts"]
    for j, v in enumerate(report.claim_verdicts):
        if j < len(expected):
            all_pairs.append({
                "claim": v.claim[:50],
                "score": v.best_score,
                "expected_supported": expected[j] == "SUPPORTED",
            })

print("Threshold Sensitivity Analysis")
print("=" * 60)
print(f"{'Threshold':>10} {'Supported':>10} {'Unsup':>8} {'Accuracy':>10}")
print("-" * 40)

for threshold in [0.40, 0.45, 0.50, 0.55, 0.60, 0.65, 0.70, 0.75, 0.80]:
    correct = 0
    supported_count = 0
    for p in all_pairs:
        predicted_supported = p["score"] >= threshold
        if predicted_supported:
            supported_count += 1
        if predicted_supported == p["expected_supported"]:
            correct += 1
    total = len(all_pairs)
    acc = correct / total
    print(f"{threshold:>10.2f} {supported_count:>10} {total-supported_count:>8} {acc:>10.1%}")

print(f"\nNote: The optimal threshold depends on whether you prefer")
print(f"catching hallucinations (lower threshold) vs avoiding false alarms (higher threshold).")


## Similarity Score Distribution

Visualize the score distribution for supported vs unsupported claims.
Good separation means the verifier can reliably distinguish them.


In [ ]:
supported_scores = [p["score"] for p in all_pairs if p["expected_supported"]]
unsupported_scores = [p["score"] for p in all_pairs if not p["expected_supported"]]

print("Score Distribution")
print("=" * 50)
print(f"\nSupported claims ({len(supported_scores)}):")
print(f"  min={min(supported_scores):.3f}  max={max(supported_scores):.3f}  "
      f"mean={np.mean(supported_scores):.3f}  std={np.std(supported_scores):.3f}")

if unsupported_scores:
    print(f"\nUnsupported claims ({len(unsupported_scores)}):")
    print(f"  min={min(unsupported_scores):.3f}  max={max(unsupported_scores):.3f}  "
          f"mean={np.mean(unsupported_scores):.3f}  std={np.std(unsupported_scores):.3f}")
else:
    print("\n  No unsupported claims in test set.")

sep = np.mean(supported_scores) - (np.mean(unsupported_scores) if unsupported_scores else 0)
print(f"\nMean separation: {sep:.3f}")
if sep > 0.15:
    print("  Good separation - verifier can reliably distinguish support from hallucination.")
elif sep > 0.05:
    print("  Moderate separation - some overlap expected, threshold tuning important.")
else:
    print("  Poor separation - embedding model may not be sufficient for this task.")

# Show per-claim breakdown
print(f"\nPer-claim scores:")
for p in all_pairs:
    label = "SUPPORTED" if p["expected_supported"] else "HALLUCIN."
    bar = "#" * int(p["score"] * 30)
    print(f"  {label:>10s} | {p['score']:.3f} {bar} {p['claim']}")


## Error Analysis

Examine cases where the verifier made mistakes.
Understanding failure modes helps improve the system.


In [ ]:
print("Error Analysis")
print("=" * 60)

errors_found = False
for report, scenario in zip(all_reports, test_scenarios):
    expected = scenario["expected_verdicts"]
    for j, v in enumerate(report.claim_verdicts):
        if j >= len(expected):
            break
        exp_supported = expected[j] == "SUPPORTED"
        act_supported = v.verdict in ("SUPPORTED", "PARTIAL")
        
        if exp_supported != act_supported:
            errors_found = True
            error_type = "FALSE NEGATIVE" if exp_supported else "FALSE POSITIVE"
            print(f"\n  {error_type}:")
            print(f"    Claim: {v.claim}")
            print(f"    Expected: {expected[j]}, Got: {v.verdict} (score={v.best_score:.3f})")
            print(f"    Best chunk: [{v.best_chunk_id}] {v.best_chunk_text[:80]}...")
            
            if error_type == "FALSE NEGATIVE":
                print(f"    Diagnosis: Supported claim scored too low.")
                print(f"    Fix: Lower threshold or use cross-encoder reranking.")
            else:
                print(f"    Diagnosis: Hallucinated claim matched a chunk too closely.")
                print(f"    Fix: Raise threshold or use NLI-based verification.")

if not errors_found:
    print("  No errors found - all verdicts matched expected labels.")

# Common hallucination patterns
print(f"\nCommon Hallucination Patterns in RAG:")
patterns = [
    ("Entity confusion", "Mixing facts about similar entities (e.g., Europa vs Enceladus)"),
    ("Fabricated statistics", "Inventing numbers not in any retrieved chunk"),
    ("Temporal fabrication", "Adding dates or timelines not in context"),
    ("Causal hallucination", "Inferring cause-effect not stated in evidence"),
    ("Scope creep", "Answering beyond what the retrieved chunks cover"),
]
for i, (name, desc) in enumerate(patterns, 1):
    print(f"  {i}. {name}: {desc}")


## Limitations

1. **Embedding similarity is not entailment.** Two sentences can be
   semantically similar without one entailing the other. An NLI model
   (e.g., cross-encoder trained on MNLI) would be more precise.

2. **Sentence-level granularity.** Some claims contain multiple facts;
   one part may be supported while another is not.

3. **No negation handling.** "Mars has no water" and "Mars has water"
   are semantically similar but factually opposite.

4. **Threshold sensitivity.** The boundary between SUPPORTED and
   PARTIAL depends on the embedding model and domain.

5. **Pre-written answers.** In production, answers are generated by
   an LLM. LLM-generated text has different hallucination patterns
   than hand-written examples.

6. **Small corpus.** With 25 chunks, most topics have exactly one
   relevant chunk. Larger corpora create more ambiguity.


## Better Approach: NLI-Based Verification

In production, use a **Natural Language Inference (NLI)** model instead
of raw embedding similarity. NLI models are trained to determine whether
a hypothesis (claim) is **entailed by**, **contradicted by**, or
**neutral to** a premise (chunk).

```
Premise:  "Mars has two small moons named Phobos and Deimos."
Hypothesis: "Mars has three moons."

Embedding similarity: HIGH (both about Mars moons)
NLI prediction: CONTRADICTION (two != three)
```

Recommended models:
- `cross-encoder/nli-deberta-v3-base` -- strong NLI model
- `MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli` -- trained on
  multiple NLI datasets

The trade-off: NLI models are slower (need claim x chunk pairs)
but much more accurate at detecting contradictions and partial support.


## Common Mistakes

| Mistake | Why it's wrong | Fix |
|---------|---------------|-----|
| Verifying against ALL corpus chunks | Slow and noisy; irrelevant chunks add false matches | Verify only against retrieved chunks |
| Using embedding sim as entailment | Similar != supported (negation problem) | Use NLI model for precision |
| Binary verdict only | Loses nuance between partial and no support | Use 3-level scale: S/P/U |
| Skipping claim extraction | Whole-answer verification misses individual hallucinations | Break into granular claims |
| Fixed threshold everywhere | Different domains need different calibration | Tune threshold on labeled data |
| Not logging verdicts | Can't improve without data | Log claim + verdict + score for analysis |


## Mini Challenge

1. **NLI-based verification.** Replace embedding similarity with
   `cross-encoder/nli-deberta-v3-base`. Compare accuracy on the same
   test scenarios.

2. **Negation test.** Add test scenarios with negated claims
   ("Mars has no moons" when the chunk says it has two).
   Does the embedding approach catch it?

3. **LLM-as-judge.** Use an LLM to verify each claim against chunks.
   Compare cost, latency, and accuracy vs embedding/NLI approaches.

4. **Citation linking.** Instead of just verdict, output the exact
   sentence within the chunk that supports each claim.

5. **Combine with query rewriting.** Use Project 22's rewriting
   to improve retrieval quality, then verify the answer.
   Does better retrieval reduce hallucination?


## Production Considerations

| Aspect | Approach |
|--------|----------|
| **Latency** | Embedding verification adds ~50ms; NLI adds ~200ms per claim |
| **Threshold tuning** | Calibrate on labeled data from your domain |
| **Fallback** | If groundedness < 50%, regenerate with stricter prompt |
| **User experience** | Show confidence indicators next to each claim |
| **Monitoring** | Track groundedness score over time to catch model drift |
| **Human review** | Route low-groundedness answers to human reviewers |
| **Caching** | Cache claim-chunk similarity for repeated questions |
| **Scaling** | Batch encode claims; use FAISS for large chunk sets |


## How to Improve This Project

1. **NLI model for verification** -- Handles negation and contradiction.
2. **LLM-powered claim extraction** -- More precise than sentence splitting.
3. **Sub-sentence claims** -- Break compound sentences into atomic facts.
4. **Confidence calibration** -- Map similarity scores to actual probabilities.
5. **Feedback loop** -- Let users flag wrong verdicts to improve thresholds.
6. **Multi-step verification** -- First retrieve, then verify, then re-retrieve
   if groundedness is low (combine with Project 23's agentic approach).
7. **Chunk-level attribution** -- Highlight the exact span in each chunk
   that supports the claim.


## Key Takeaways

1. **RAG doesn't eliminate hallucination.** The generator can still invent
   facts not in the retrieved chunks. Citation verification is essential.

2. **Break answers into claims.** Whole-answer verification misses
   individual hallucinated sentences. Claim-level checking is granular.

3. **Embedding similarity is a reasonable first pass.** It catches
   obvious hallucinations (score < 0.5) with low latency.

4. **NLI models are more precise.** They handle negation, contradiction,
   and partial support better than raw embedding distance.

5. **Threshold tuning matters.** The boundary between supported and
   unsupported depends on your domain and tolerance for false alarms.

6. **Verification is cheap insurance.** A few hundred milliseconds to
   check groundedness is far cheaper than serving wrong answers.

7. **Composable with other RAG techniques.** Better retrieval (Project 21-23)
   + verification = production-grade trustworthy RAG.
